# Projet 3 — Backtesting d'une stratégie systématique
## Notebook 08a-bis — Univers S&P 500 complet (avec les 8 titres du momentum)

**Rôle.** On reprend l'univers S&P 500 élargi et on y **ajoute les 8 titres du momentum** (AAPL, MSFT, NVDA, AMZN, JPM, XOM, JNJ, PG), rangés dans leurs secteurs. Objectif : ne priver le screening d'aucun candidat, notamment les paires bancaires avec JPM et pétrolières avec XOM.

Les 8 titres sont classés ainsi : JPM→Banks, XOM→Oil_Gas, NVDA→Semis, AMZN→Retail_Big, JNJ→Pharma, PG→Consumer_Staples, AAPL+MSFT→Tech_Hardware.

**Alignement (option b).** On garde tous les titres et on aligne sur leur date commune. On retire seulement les titres démarrant après 2012 pour ne pas trop raccourcir l'historique. La période sera proche de celle de l'univers S&P 500 précédent (~fin 2011 → 2026).

> À exécuter sur ta machine (yfinance). Archivage `.parquet`.


In [1]:
import os
import numpy as np, pandas as pd
try:
    import yfinance as yf
except ImportError:
    raise ImportError("Installe yfinance : pip install yfinance")
DATA_DIR="../data"; os.makedirs(DATA_DIR, exist_ok=True)

SECTORS = {
 "Banks": ["BAC","WFC","C","GS","MS","USB","PNC","TFC","COF","BK","STT","FITB","HBAN","RF","CFG","KEY","MTB","NTRS","ALLY","JPM"],
 "Insurance": ["BRK-B","PGR","TRV","ALL","MET","PRU","AIG","CB","AFL","HIG","PFG","AJG","MMC","AON","CINF","L"],
 "CapitalMarkets": ["SPGI","MCO","ICE","CME","MSCI","NDAQ","SCHW","BLK","TROW","AMP"],
 "Oil_Gas": ["CVX","COP","SLB","EOG","MPC","PSX","VLO","OXY","HES","DVN","FANG","HAL","BKR","WMB","KMI","OKE","TRGP","XOM"],
 "Utilities": ["NEE","DUK","SO","D","AEP","EXC","SRE","XEL","ED","WEC","ES","PEG","AEE","DTE","PPL","CMS","CNP","ATO"],
 "REITs": ["AMT","PLD","CCI","EQIX","PSA","O","SPG","WELL","DLR","VICI","AVB","EQR","SBAC","ARE","VTR","ESS","MAA"],
 "Semis": ["AMD","INTC","QCOM","TXN","MU","ADI","AMAT","LRCX","KLAC","MCHP","NXPI","ON","MPWR","NVDA"],
 "Software": ["ORCL","CRM","ADBE","NOW","INTU","IBM","ADP","FIS","FI","CTSH","ACN"],
 "Tech_Hardware": ["AAPL","MSFT"],
 "Pharma": ["LLY","ABBV","MRK","PFE","BMY","AMGN","GILD","VRTX","REGN","BIIB","ZTS","JNJ"],
 "HealthEquip": ["ABT","TMO","DHR","MDT","SYK","BSX","BDX","ISRG","EW","ZBH","BAX"],
 "Payments_Cards": ["V","MA","AXP","PYPL","GPN"],
 "Telecom": ["VZ","T","TMUS"],
 "Retail_Big": ["HD","LOW","TGT","WMT","COST","DG","DLTR","BBY","AMZN"],
 "Consumer_Staples": ["KO","PEP","MDLZ","CL","KMB","GIS","KHC","HSY","STZ","K","SYY","ADM","PG"],
 "Food_Restaurant": ["MCD","SBUX","CMG","YUM","DRI"],
 "Industrials_Machinery": ["CAT","DE","HON","GE","MMM","EMR","ITW","ETN","PH","ROK","DOV","CMI"],
 "Aerospace_Defense": ["BA","LMT","RTX","NOC","GD","LHX","TDG","HWM"],
 "Rails_Transport": ["UNP","CSX","NSC","FDX","UPS","ODFL"],
 "Autos": ["TSLA","F","GM","APTV","BWA"],
 "Chemicals": ["LIN","APD","SHW","ECL","DD","DOW","PPG","NEM","FCX"],
 "Media_Comm": ["GOOGL","GOOG","META","NFLX","DIS","CMCSA","CHTR","WBD","TTWO","EA"],
}
TICKERS = sorted({t for v in SECTORS.values() for t in v})
print(f"{len(TICKERS)} titres, {len(SECTORS)} secteurs (les 8 du momentum inclus).")

# Même fenêtre que le momentum ; l'alignement (option b) fixera le début effectif
START="2010-01-01"; END="2026-08-13"   # fin calée juste après le momentum (2026-08-12)
CUTOFF="2012-01-01" 

234 titres, 22 secteurs (les 8 du momentum inclus).


In [2]:
RAW_PATH = os.path.join(DATA_DIR, "sp500full_prices_raw.parquet")

def download_prices(tickers, start, end, path, force_download=False):
    if os.path.exists(path) and not force_download:
        print(f"Chargement du cache local : {path}")
        return pd.read_parquet(path)
    print(f"Téléchargement de {len(tickers)} titres via yfinance (1-2 min)...")
    raw = yf.download(tickers, start=start, end=end, auto_adjust=False,
                      group_by="column", progress=False, threads=True)
    if raw.empty: raise RuntimeError("Téléchargement vide.")
    raw.to_parquet(path)
    print(f"Archivé : {path}  (shape={raw.shape})")
    return raw

raw = download_prices(TICKERS, START, END, RAW_PATH)

Téléchargement de 234 titres via yfinance (1-2 min)...


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: HES"}}}
$MMC: possibly delisted; no timezone found
$K: possibly delisted; no timezone found
$HES: possibly delisted; no timezone found
$FI: possibly delisted; no timezone found
$BK: possibly delisted; no timezone found

5 Failed downloads:
['MMC', 'K', 'HES', 'FI', 'BK']: possibly delisted; no timezone found


Archivé : ../data\sp500full_prices_raw.parquet  (shape=(4177, 1404))


In [3]:
def get_field(raw, field):
    if isinstance(raw.columns, pd.MultiIndex): return raw[field].copy()
    return raw[[field]].copy()

adj_raw = get_field(raw, "Adj Close")
missing = [t for t in TICKERS if t not in adj_raw.columns or adj_raw[t].notna().sum() == 0]
print("Titres non récupérés (ignorés) :", missing if missing else "aucun")
adj_raw = adj_raw[[c for c in adj_raw.columns if c not in missing]]

starts = adj_raw.apply(lambda s: s.first_valid_index())
late = starts[starts > CUTOFF].sort_values()
print(f"\nTitres démarrant après {CUTOFF} (retirés) :")
print(late.to_string() if len(late) else "  aucun")
adj_raw = adj_raw.drop(columns=late.index.tolist())
print(f"\nTitres conservés : {adj_raw.shape[1]}")

Titres non récupérés (ignorés) : ['BK', 'FI', 'HES', 'K', 'MMC']

Titres démarrant après 2012-01-01 (retirés) :
Ticker
PSX    2012-04-12
META   2012-05-18
NOW    2012-06-29
FANG   2012-10-12
ABBV   2013-01-02
ZTS    2013-02-01
ALLY   2014-01-28
CFG    2014-09-24
KHC    2015-07-06
PYPL   2015-07-06
HWM    2016-11-01
VICI   2018-01-02
DOW    2019-03-20
EA     2026-07-17

Titres conservés : 215


In [4]:
def clean_prices(df, ffill_limit=5):
    df = df[~df.index.duplicated(keep="first")].sort_index()
    df = df.ffill(limit=ffill_limit)
    df = df.dropna(how="any")
    return df

adj_clean = clean_prices(adj_raw)
print("Shape après nettoyage/alignement :", adj_clean.shape)
print(f"Période effective : {adj_clean.index.min().date()} -> {adj_clean.index.max().date()}")
print("Valeurs manquantes :", int(adj_clean.isna().sum().sum()))

# Contrôle : les 8 du momentum sont-ils bien présents ?
mom = ["AAPL","MSFT","NVDA","AMZN","JPM","XOM","JNJ","PG"]
print("Titres momentum présents :", [t for t in mom if t in adj_clean.columns])

Shape après nettoyage/alignement : (3702, 215)
Période effective : 2011-11-17 -> 2026-08-12
Valeurs manquantes : 0
Titres momentum présents : ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'JPM', 'XOM', 'JNJ', 'PG']


In [5]:
adj_clean.to_csv(os.path.join(DATA_DIR, "sp500full_adj_close.csv"))
sector_map = {t: sec for sec, ts in SECTORS.items() for t in ts if t in adj_clean.columns}
pd.Series(sector_map, name="secteur").rename_axis("ticker").to_csv(
    os.path.join(DATA_DIR, "sp500full_sectors.csv"))

from itertools import combinations
kept = pd.Series(sector_map)
npairs = sum(len(list(combinations(kept[kept==s].index, 2))) for s in kept.unique())
print(f"Titres conservés : {adj_clean.shape[1]} | paires intra-secteur : {npairs}")
for f in ["sp500full_prices_raw.parquet","sp500full_adj_close.csv","sp500full_sectors.csv"]:
    p=os.path.join(DATA_DIR,f)
    if os.path.exists(p): print(f"  - {f}  ({os.path.getsize(p)/1024:.0f} Ko)")
print("\nEnvoie-moi sp500full_adj_close.csv pour lancer le screening complet.")

Titres conservés : 215 | paires intra-secteur : 1170
  - sp500full_prices_raw.parquet  (35577 Ko)
  - sp500full_adj_close.csv  (14204 Ko)
  - sp500full_sectors.csv  (3 Ko)

Envoie-moi sp500full_adj_close.csv pour lancer le screening complet.
